# Donovan Enterprises Production Planning with AMPL in Python
## AMPL Modeling for Problem 4

## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

```python
%pip install amplpy
```

In [1]:
from __future__ import annotations

from functools import wraps
from time import perf_counter

In [2]:
%pip install amplpy
from amplpy import AMPL, modules

modules.install('highs')


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
def create_ampl_instance(solver: str = "highs"):
    ampl = AMPL()
    ampl.option["solver"] = solver
    ampl.option["solver_msg"] = 0
    return ampl

## Problem: Donovan Enterprises Blender Production

Donovan must meet blender demand over 4 quarters:
- Q1: 4,000 | Q2: 2,000 | Q3: 3,000 | Q4: 10,000

**Workforce:**
- Each employee works 3 of 4 quarters (1 quarter off)
- 4 possible schedules: off Q1, off Q2, off Q3, off Q4
- Annual salary: $30,000 per employee
- Each working employee produces up to 500 blenders/quarter

**Inventory:**
- Holding cost: $30/unit/quarter
- Initial inventory: 600 units

Minimize total cost (labor + inventory).

In [4]:
SCHEDULES = ["A", "B", "C", "D"]

WORKS_IN = {
    ("A", 1): 1, ("A", 2): 1, ("A", 3): 1, ("A", 4): 0,
    ("B", 1): 1, ("B", 2): 1, ("B", 3): 0, ("B", 4): 1,
    ("C", 1): 1, ("C", 2): 0, ("C", 3): 1, ("C", 4): 1,
    ("D", 1): 0, ("D", 2): 1, ("D", 3): 1, ("D", 4): 1,
}

DEMAND = {1: 4000, 2: 2000, 3: 3000, 4: 10000}
INIT_INV = 600
SALARY = 30000
HOLDING_COST = 30
CAPACITY = 500

In [5]:
@timed
def solve_donovan(solver: str = "highs"):
    ampl = create_ampl_instance(solver)

    ampl.eval(
        r'''
        set SCHEDULES ordered;
        set QUARTERS = 1..4;

        param works_in {SCHEDULES, QUARTERS} binary;
        param demand {QUARTERS} >= 0;
        param init_inv >= 0;
        param salary >= 0;
        param holding_cost >= 0;
        param capacity >= 0;

        var Employees {SCHEDULES} >= 0;
        var Production {QUARTERS} >= 0;
        var Inventory {0..4} >= 0;

        minimize TotalCost:
            salary * sum {s in SCHEDULES} Employees[s]
            + holding_cost * sum {t in QUARTERS} Inventory[t];

        subject to InitialInventory:
            Inventory[0] = init_inv;

        subject to InvBalance {t in QUARTERS}:
            Inventory[t] = Inventory[t-1] + Production[t] - demand[t];

        subject to ProdCapacity {t in QUARTERS}:
            Production[t] <= capacity * sum {s in SCHEDULES} works_in[s, t] * Employees[s];
        '''
    )

    ampl.set["SCHEDULES"] = SCHEDULES
    ampl.param["works_in"] = WORKS_IN
    ampl.param["demand"] = DEMAND
    ampl.param["init_inv"] = INIT_INV
    ampl.param["salary"] = SALARY
    ampl.param["holding_cost"] = HOLDING_COST
    ampl.param["capacity"] = CAPACITY

    ampl.solve(solver=solver)

    emp_vals = ampl.get_variable("Employees").get_values().to_dict()
    prod_vals = ampl.get_variable("Production").get_values().to_dict()
    inv_vals = ampl.get_variable("Inventory").get_values().to_dict()

    solution = {
        "employees": {},
        "production": {},
        "inventory": {},
        "total_cost": round(ampl.get_value("TotalCost"), 2),
    }

    for k, v in emp_vals.items():
        key = k if not isinstance(k, tuple) else k[0]
        solution["employees"][key] = round(v, 2)

    for k, v in prod_vals.items():
        key = k if not isinstance(k, tuple) else k[0]
        solution["production"][key] = round(v, 2)

    for k, v in inv_vals.items():
        key = k if not isinstance(k, tuple) else k[0]
        solution["inventory"][key] = round(v, 2)

    return solution

In [6]:
result, elapsed = solve_donovan()

print("=== DONOVAN ENTERPRISES RESULTS ===")
print(f"\nEmployees by schedule:")
schedule_desc = {"A": "off Q4", "B": "off Q3", "C": "off Q2", "D": "off Q1"}
total_emp = 0
for s in SCHEDULES:
    val = result["employees"].get(s, 0)
    total_emp += val
    print(f"  Schedule {s} ({schedule_desc[s]}): {val:.2f}")
print(f"  Total employees: {total_emp:.2f}")

print(f"\nProduction and inventory by quarter:")
print(f"{'Quarter':<10} {'Demand':>8} {'Production':>12} {'Inventory':>12}")
print("-" * 44)
for t in range(1, 5):
    prod = result["production"].get(t, 0)
    inv = result["inventory"].get(t, 0)
    print(f"Q{t:<9} {DEMAND[t]:>8} {prod:>12.2f} {inv:>12.2f}")

print(f"\nLabor cost:     ${SALARY * total_emp:,.2f}")
inv_cost = HOLDING_COST * sum(result["inventory"].get(t, 0) for t in range(1, 5))
print(f"Inventory cost: ${inv_cost:,.2f}")
print(f"Total cost:     ${result['total_cost']:,.2f}")
print(f"Time:           {elapsed:.8f} seconds")

HiGHS 1.11.0=== DONOVAN ENTERPRISES RESULTS ===

Employees by schedule:
  Schedule A (off Q4): 0.00
  Schedule B (off Q3): 0.00
  Schedule C (off Q2): 6.80
  Schedule D (off Q1): 6.20
  Total employees: 13.00

Production and inventory by quarter:
Quarter      Demand   Production    Inventory
--------------------------------------------
Q1             4000      3400.00         0.00
Q2             2000      2000.00         0.00
Q3             3000      6500.00      3500.00
Q4            10000      6500.00         0.00

Labor cost:     $390,000.00
Inventory cost: $105,000.00
Total cost:     $495,000.00
Time:           0.01222122 seconds


## Sensitivity Analysis

In [7]:
ampl = create_ampl_instance()

ampl.eval(
    r'''
    set SCHEDULES ordered;
    set QUARTERS = 1..4;

    param works_in {SCHEDULES, QUARTERS} binary;
    param demand {QUARTERS} >= 0;
    param init_inv >= 0;
    param salary >= 0;
    param holding_cost >= 0;
    param capacity >= 0;

    var Employees {SCHEDULES} >= 0;
    var Production {QUARTERS} >= 0;
    var Inventory {0..4} >= 0;

    minimize TotalCost:
        salary * sum {s in SCHEDULES} Employees[s]
        + holding_cost * sum {t in QUARTERS} Inventory[t];

    subject to InitialInventory:
        Inventory[0] = init_inv;

    subject to InvBalance {t in QUARTERS}:
        Inventory[t] = Inventory[t-1] + Production[t] - demand[t];

    subject to ProdCapacity {t in QUARTERS}:
        Production[t] <= capacity * sum {s in SCHEDULES} works_in[s, t] * Employees[s];
    '''
)

ampl.set["SCHEDULES"] = SCHEDULES
ampl.param["works_in"] = WORKS_IN
ampl.param["demand"] = DEMAND
ampl.param["init_inv"] = INIT_INV
ampl.param["salary"] = SALARY
ampl.param["holding_cost"] = HOLDING_COST
ampl.param["capacity"] = CAPACITY

ampl.solve()

print("=== SHADOW PRICES ===")
print(f"\nInventory balance constraints:")
for t in range(1, 5):
    con = ampl.get_constraint("InvBalance").get(t)
    print(f"  Q{t} shadow price: {con.dual():.4f}  slack: {con.slack():.4f}")

print(f"\nProduction capacity constraints:")
for t in range(1, 5):
    con = ampl.get_constraint("ProdCapacity").get(t)
    print(f"  Q{t} shadow price: {con.dual():.4f}  slack: {con.slack():.4f}")

print(f"\nWorkers available per quarter:")
for t in range(1, 5):
    emp_vals = ampl.get_variable("Employees").get_values().to_dict()
    workers = sum(
        WORKS_IN.get((s, t), 0) * emp_vals.get(s, emp_vals.get((s,), 0))
        for s in SCHEDULES
    )
    prod = ampl.get_variable("Production").get(t).value()
    cap = 500 * workers
    print(f"  Q{t}: {workers:.1f} workers, capacity {cap:.0f}, used {prod:.0f}")

HiGHS 1.11.0=== SHADOW PRICES ===

Inventory balance constraints:
  Q1 shadow price: 0.0000  slack: 0.0000
  Q2 shadow price: 0.0000  slack: 0.0000
  Q3 shadow price: -15.0000  slack: 0.0000
  Q4 shadow price: -45.0000  slack: 0.0000

Production capacity constraints:
  Q1 shadow price: 0.0000  slack: 0.0000
  Q2 shadow price: 0.0000  slack: 1100.0000
  Q3 shadow price: -15.0000  slack: 0.0000
  Q4 shadow price: -45.0000  slack: 0.0000

Workers available per quarter:
  Q1: 6.8 workers, capacity 3400, used 3400
  Q2: 6.2 workers, capacity 3100, used 2000
  Q3: 13.0 workers, capacity 6500, used 6500
  Q4: 13.0 workers, capacity 6500, used 6500


## Interpretation

The Donovan problem balances hiring costs against inventory holding costs:

- **Schedule allocation**: The optimal solution distributes employees across schedules to match quarterly demand peaks, particularly the large Q4 demand of 10,000 units.
- **Inventory build-up**: It may be cheaper to produce extra units in earlier quarters (building inventory) than to hire additional employees just for Q4.
- **Production capacity constraints**: Shadow prices on capacity constraints show in which quarters additional workforce would be most valuable.
- **Inventory shadow prices**: Show the marginal cost of having to meet demand in each quarter — higher values indicate tighter quarters.